# Cookie Recipe Adaptation Agent

This notebook builds a single AI agent designed to help users adapt cookie recipes. The agent focuses on common cookie-baking tasks such as ingredient substitutions, batch size changes, dietary adjustments, and texture guidance.

The system uses retrieval-augmented generation (RAG), a small cookie baking knowledge base, multiple custom tools, and a LangChain agent workflow. Together, these components help the agent provide more grounded and practical responses instead of relying only on general language model knowledge.

## Main Features
- retrieve cookie recipe and baking guidance context
- scale recipe ingredients up or down
- suggest ingredient substitutions with likely baking effects
- adapt a full cookie recipe for a requested change
- stay within a focused cookie-only scope

## Recommended Run Order
Run the notebook from Section 1 through Section 8 for the core project build and testing.

- Sections 1–5: data preparation, knowledge base creation, and retriever setup
- Section 6: custom tool definitions
- Section 7: single-agent setup with LangChain
- Section 8: real-world scenario testing
- Section 9: optional Gradio interface
- Section 10: final notes and limitations

## Project Scope
This project is intentionally limited to cookie recipes so the agent stays focused, practical, and more reliable for the final project.
## Environment Note

This notebook was developed in Google Colab and uses Google Drive for file storage.  
Some file paths assume a project folder in `/content/drive/MyDrive/ITAI2376_CookieAgent/`.

# Section 1 — Loading and inspecting the raw recipe dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd

file_path = "/content/drive/MyDrive/ITAI2376_CookieAgent/datasets/recipes_ingredients.csv"
df = pd.read_csv(file_path)

print("Column names:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

Column names:
['id', 'name', 'description', 'ingredients', 'ingredients_raw', 'steps', 'servings', 'serving_size', 'tags']

First 5 rows:


,id,name,description,ingredients,ingredients_raw,steps,servings,serving_size,tags
0,71247,Cherry Streusel Cobbler,"I haven't made this in years, so I'm just gues...","[""cherry pie filling"", ""condensed milk"", ""melt...","[""2 (21 ounce) cans cherry pie filling"",""2...","[""Preheat oven to 375°F."", ""Spread cherry pie ...",6.0,1 (347 g),"[""60-minutes-or-less"", ""time-to-make"", ""course..."
1,76133,Reuben and Swiss Casserole Bake,I think this is even better than a reuben sand...,"[""corned beef chopped"", ""sauerkraut cold water...","[""1/2-1 lb corned beef, cooked and choppe...","[""Set oven to 350 degrees F."", ""Butter a 9 x 1...",4.0,1 (207 g),"[""60-minutes-or-less"", ""time-to-make"", ""course..."
2,503816,Yam-Pecan Recipe,A lady I work with heard me taking about ZWT a...,"[""unsalted butter"", ""vegetable oil"", ""all - pu...","[""3/4 cup unsalted butter, at room tempera...","[""Preheat oven to 350°F In a mixing bowl, usi...",8.0,1 (198 g),"[""time-to-make"", ""course"", ""main-ingredient"", ..."
3,418749,Tropical Orange Layer Cake,An easy and delicious cake. Great for a summ...,"[""orange cake mix"", ""instant vanilla pudding"",...","[""1 (18 ounce) pkge.orange cake mix"",""1 (3...","[""In a large mixing bowl, combine the first 6 ...",16.0,1 (191 g),"[""60-minutes-or-less"", ""time-to-make"", ""course..."
4,392934,Safe to Eat Raw Chocolate Chip Oreo Cookie &qu...,I was searching the web for something like thi...,"[""butter"", ""brown sugar"", ""granulated sugar"", ...","[""1/2 cup butter, room temperature "",""1/2 ...","[""Cream butter and sugars together."", ""Blend i...",24.0,1 (26 g),"[""15-minutes-or-less"", ""time-to-make"", ""course..."


# Section 2 — Cleaning the main cookie recipe dataset

**Section 2.1** - Parse the list-like columns

In [ ]:
import pandas as pd
import json
import ast
import warnings

# Load dataset
file_path = "/content/drive/MyDrive/ITAI2376_CookieAgent/datasets/recipes_ingredients.csv"
df = pd.read_csv(file_path)

list_cols = ["ingredients", "ingredients_raw", "steps", "tags"]

def parse_list_cell(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    if not isinstance(x, str):
        return []

    x = x.strip()

    # First try JSON parsing
    try:
        return json.loads(x)
    except:
        pass

    # Then try Python literal parsing, while suppressing warnings
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            return ast.literal_eval(x)
    except:
        return []

for col in list_cols:
    df[col] = df[col].apply(parse_list_cell)

print("Dataset loaded and list columns parsed.")
print("Shape:", df.shape)
display(df.head())

Dataset loaded and list columns parsed.
Shape: (500471, 9)


,id,name,description,ingredients,ingredients_raw,steps,servings,serving_size,tags
0,71247,Cherry Streusel Cobbler,"I haven't made this in years, so I'm just gues...","[cherry pie filling, condensed milk, melted ma...","[2 (21 ounce) cans cherry pie filling, 2 ...","[Preheat oven to 375°F., Spread cherry pie fil...",6.0,1 (347 g),"[60-minutes-or-less, time-to-make, course, mai..."
1,76133,Reuben and Swiss Casserole Bake,I think this is even better than a reuben sand...,"[corned beef chopped, sauerkraut cold water, s...","[1/2-1 lb corned beef, cooked and chopped...","[Set oven to 350 degrees F., Butter a 9 x 13-i...",4.0,1 (207 g),"[60-minutes-or-less, time-to-make, course, mai..."
2,503816,Yam-Pecan Recipe,A lady I work with heard me taking about ZWT a...,"[unsalted butter, vegetable oil, all - purpose...","[3/4 cup unsalted butter, at room temperat...","[Preheat oven to 350°F In a mixing bowl, usin...",8.0,1 (198 g),"[time-to-make, course, main-ingredient, cuisin..."
3,418749,Tropical Orange Layer Cake,An easy and delicious cake. Great for a summ...,"[orange cake mix, instant vanilla pudding, ora...","[1 (18 ounce) pkge.orange cake mix, 1 (3 ...","[In a large mixing bowl, combine the first 6 i...",16.0,1 (191 g),"[60-minutes-or-less, time-to-make, course, pre..."
4,392934,Safe to Eat Raw Chocolate Chip Oreo Cookie &qu...,I was searching the web for something like thi...,"[butter, brown sugar, granulated sugar, milk, ...","[1/2 cup butter, room temperature , 1/2 c...","[Cream butter and sugars together., Blend in m...",24.0,1 (26 g),"[15-minutes-or-less, time-to-make, course, mai..."


In [ ]:
for col in list_cols:
    print(f"{col}:")
    print(df[col].iloc[0])
    print(type(df[col].iloc[0]))
    print("-" * 40)

ingredients:
['cherry pie filling', 'condensed milk', 'melted margarine', 'cinnamon', 'nutmeg', 'light brown sugar', 'flour', 'margarine', 'chopped nuts', 'oats', 'butter - flavored cooking spray']
<class 'list'>
----------------------------------------
ingredients_raw:
['2 (21   ounce) cans   cherry pie filling', '2       eggs', '1 (14   ounce) can   sweetened condensed milk (not evaporated)', '1/4  cup   melted margarine', '1/2  teaspoon    cinnamon', '1/4  teaspoon    nutmeg', '1/2  cup   firmly packed light brown sugar', '1/2  cup    self-rising flour', '1/4  cup    margarine', "1/2  cup   chopped nuts (chef's choice)", '1/2  cup    quick-cooking oats', '  butter-flavored cooking spray']
<class 'list'>
----------------------------------------
steps:
['Preheat oven to 375°F.', 'Spread cherry pie filling in a 8 x 8-inch square pan that has lightly been sprayed.', 'In a medium-size bowl, beat eggs.', 'Add milk, melted margarine, cinnamon, and nutmeg; mix well.', 'Pour over cherry pie 

**Section 2.2** - Filter to cookie recipes only

In [ ]:
# Basic cleanup
df = df.dropna(subset=["name"]).copy()
df["name_clean"] = df["name"].fillna("").str.lower().str.strip()
df["tag_text"] = df["tags"].apply(lambda x: " | ".join([str(i).lower() for i in x]))

# Must mention cookie/cookies
include_mask = (
    df["name_clean"].str.contains(r"\bcookies?\b", regex=True, na=False) |
    df["tag_text"].str.contains(r"\bcookies?\b", regex=True, na=False)
)

# Exclude common non-cookie bar types
exclude_terms = r"\bbars?\b|\blemon bars?\b|\bmagic bars?\b|\bcheesecake bars?\b|\bbrownie bars?\b"
exclude_mask = (
    df["name_clean"].str.contains(exclude_terms, regex=True, na=False) |
    df["tag_text"].str.contains(exclude_terms, regex=True, na=False)
)

cookie_df = df[include_mask & ~exclude_mask].copy()

print("Number of cookie recipes found after stricter filtering:", len(cookie_df))
display(cookie_df[["id", "name", "tags"]].head(10))

Number of cookie recipes found after stricter filtering: 22089


,id,name,tags
4,392934,Safe to Eat Raw Chocolate Chip Oreo Cookie &qu...,"[15-minutes-or-less, time-to-make, course, mai..."
44,148423,Airbake Giant Cashew White Chocolate Cookies,"[60-minutes-or-less, time-to-make, course, pre..."
58,444972,Chocolate Chip Shortbread,"[60-minutes-or-less, time-to-make, course, mai..."
66,343158,Maple Meltaways,"[60-minutes-or-less, time-to-make, course, cui..."
85,457289,Somebody's Mother's Buttercream Icing,"[15-minutes-or-less, time-to-make, course, pre..."
103,319332,Cinna-Raisin-Chips.,"[15-minutes-or-less, time-to-make, course, pre..."
146,118078,Where's Mom's Chocolate Chip Apricot Cookies?,"[lactose, 15-minutes-or-less, time-to-make, co..."
215,530773,Joan's &quot;Missing You&quot; Blondies,"[time-to-make, course, preparation, desserts, ..."
220,469674,Chewy Gingersnaps With Variations,"[60-minutes-or-less, time-to-make, course, pre..."
248,295520,Toronto Star Monster Cookies,"[30-minutes-or-less, time-to-make, course, mai..."


**Section 2.3** - Keep only columns I actually need

In [ ]:
# Keep only the columns we actually need
cookie_df = cookie_df[
    ["id", "name", "description", "ingredients", "ingredients_raw", "steps", "servings", "serving_size", "tags"]
].copy()

# Create helper text version of ingredients_raw so duplicates can be checked safely
def list_to_text(x):
    if isinstance(x, list):
        return " | ".join([str(i).strip() for i in x])
    return ""

cookie_df["ingredients_raw_key"] = cookie_df["ingredients_raw"].apply(list_to_text)

# Remove duplicates based on recipe name + raw ingredients text
cookie_df = cookie_df.drop_duplicates(subset=["name", "ingredients_raw_key"]).reset_index(drop=True)

print("Shape after selecting useful columns and dropping duplicates:", cookie_df.shape)
display(cookie_df.head())

Shape after selecting useful columns and dropping duplicates: (21769, 10)


,id,name,description,ingredients,ingredients_raw,steps,servings,serving_size,tags,ingredients_raw_key
0,392934,Safe to Eat Raw Chocolate Chip Oreo Cookie &qu...,I was searching the web for something like thi...,"[butter, brown sugar, granulated sugar, milk, ...","[1/2 cup butter, room temperature , 1/2 c...","[Cream butter and sugars together., Blend in m...",24.0,1 (26 g),"[15-minutes-or-less, time-to-make, course, mai...","1/2 cup butter, room temperature | 1/2 cu..."
1,148423,Airbake Giant Cashew White Chocolate Cookies,This is a recipe I found on the back of the wr...,"[butter margarine, brown sugar, vanilla, bakin...","[2/3 cup butter or 2/3 cup margarine, ...","[Preheat oven to 325., In a large bowl, cream ...",1.0,1 (1271 g),"[60-minutes-or-less, time-to-make, course, pre...","2/3 cup butter or 2/3 cup margarine, s..."
2,444972,Chocolate Chip Shortbread,I found this recipe in the Toronto Sun. Altho...,"[salted butter, icing sugar, cornstarch, semi ...","[1 lb salted butter, softened , 3/4 cup ...",[],1.0,1 (1088 g),"[60-minutes-or-less, time-to-make, course, mai...","1 lb salted butter, softened | 3/4 cup ..."
3,343158,Maple Meltaways,"Ohhh, these delicate cookies are always sought...","[butter, granulated sugar, maple syrup, maple ...","[1 1/2 cups unsalted butter, 1 1/3 cups ...","[Cream together butter and sugar., Add maple s...",1.0,1 (1705 g),"[60-minutes-or-less, time-to-make, course, cui...",1 1/2 cups unsalted butter | 1 1/3 cups ...
4,457289,Somebody's Mother's Buttercream Icing,Our family favorite buttercream icing recipe,"[confectioners sugar, mexican vanilla]","[1/2 cup butter, 1/4 teaspoon salt, 2 ...",[],10.0,1 (46 g),"[15-minutes-or-less, time-to-make, course, pre...",1/2 cup butter | 1/4 teaspoon salt | 2...


**Section 2.4** - Create clean text fields for retrieval later

In [ ]:
def join_list(items):
    if not isinstance(items, list):
        return ""
    return " | ".join([str(i).strip() for i in items])

cookie_df["ingredients_text"] = cookie_df["ingredients"].apply(join_list)
cookie_df["ingredients_raw_text"] = cookie_df["ingredients_raw"].apply(join_list)
cookie_df["steps_text"] = cookie_df["steps"].apply(join_list)
cookie_df["tags_text"] = cookie_df["tags"].apply(join_list)

# Combined text field for retrieval
cookie_df["retrieval_text"] = (
    "Recipe Name: " + cookie_df["name"].fillna("") + "\n" +
    "Description: " + cookie_df["description"].fillna("") + "\n" +
    "Ingredients: " + cookie_df["ingredients_raw_text"].fillna("") + "\n" +
    "Steps: " + cookie_df["steps_text"].fillna("") + "\n" +
    "Tags: " + cookie_df["tags_text"].fillna("")
)

display(cookie_df[["name", "retrieval_text"]].head(3))

,name,retrieval_text
0,Safe to Eat Raw Chocolate Chip Oreo Cookie &qu...,Recipe Name: Safe to Eat Raw Chocolate Chip Or...
1,Airbake Giant Cashew White Chocolate Cookies,Recipe Name: Airbake Giant Cashew White Chocol...
2,Chocolate Chip Shortbread,Recipe Name: Chocolate Chip Shortbread\nDescri...


**Section 2.5** - Save the cleaned cookie dataset

In [ ]:
output_path = "/content/drive/MyDrive/ITAI2376_CookieAgent/cleaned_data/cookie_recipes_clean.csv"
cookie_df.to_csv(output_path, index=False)

print(f"Saved cleaned cookie dataset to: {output_path}")
print("Final shape:", cookie_df.shape)

Saved cleaned cookie dataset to: /content/drive/MyDrive/ITAI2376_CookieAgent/cleaned_data/cookie_recipes_clean.csv
Final shape: (21769, 15)


In [ ]:
print("Sample cookie recipe names:")
display(cookie_df["name"].sample(30, random_state=42).reset_index(drop=True))

Sample cookie recipe names:


,name
0,Cranberry Cookies
1,Chocolate Walnut Cookies
2,Brown-Edged Butter Cookies
3,Vegan Macaroons
4,Flax Oatmeal Chocolate Chip Cookies
5,Rocky Road Trippers
6,Suzanne's Chocolate-Chocolate Chip Butter Ball...
7,Sugar Cookies
8,Grandma Kathryn's Meltaways
9,Jumbo Crispy Oatmeal Cookies


# Section 3 — Creating the working subset for development

Smaller working subset + first verison of the cookie knowledge base file for RAG

In [ ]:
working_cookie_df = cookie_df.sample(n=3000, random_state=42).reset_index(drop=True)

working_output_path = "/content/drive/MyDrive/ITAI2376_CookieAgent/cleaned_data/cookie_recipes_working_subset.csv"
working_cookie_df.to_csv(working_output_path, index=False)

print(f"Saved working subset to: {working_output_path}")
print("Working subset shape:", working_cookie_df.shape)

Saved working subset to: /content/drive/MyDrive/ITAI2376_CookieAgent/cleaned_data/cookie_recipes_working_subset.csv
Working subset shape: (3000, 15)


In [ ]:
preview_output_path = "/content/drive/MyDrive/ITAI2376_CookieAgent/cleaned_data/cookie_recipes_preview_100.csv"
working_cookie_df.head(100).to_csv(preview_output_path, index=False)

print(f"Saved preview file to: {preview_output_path}")

Saved preview file to: /content/drive/MyDrive/ITAI2376_CookieAgent/cleaned_data/cookie_recipes_preview_100.csv


# Section 4 — Creating the cookie baking knowledge base

This is the part that will help the agent answer things like:

* what happens if I swap butter
* why brown sugar changes texture
* what egg substitutes do
* what makes cookies chewy vs crispy

**Section 4.1** — Make the folders if needed

In [ ]:
import os

kb_dir = "/content/drive/MyDrive/ITAI2376_CookieAgent/data/knowledge_base"
os.makedirs(kb_dir, exist_ok=True)

print("Knowledge base folder ready:", kb_dir)

Knowledge base folder ready: /content/drive/MyDrive/ITAI2376_CookieAgent/data/knowledge_base


**Section 4.2** — Create the knowledge base text

In [ ]:
cookie_knowledge_text = """
# Cookie Baking Knowledge Base

## Cookie Texture Basics
Chewier cookies usually come from higher moisture and a softer center. Brown sugar, melted butter, and shorter baking can all push cookies toward a chewier result.
Crispier cookies usually come from lower-moisture formulas, more granulated sugar, and longer baking.
Small changes in sugar, fat, and bake time can shift the same cookie recipe toward chewy, crispy, or cakey results.

Sources:
- King Arthur Baking: "Cookie chemistry"
- King Arthur Baking: "Cookie science: How to achieve your perfect chocolate chip cookie"

## Butter and Fat
Butter adds flavor and affects spread. Melted butter often increases spread and can lead to denser or fudgier centers, while softened butter is commonly used for creaming because it helps incorporate air into the dough.
If cookies spread too much, warm fat, excess liquid, or inaccurate ingredient ratios may be part of the problem.

Sources:
- King Arthur Baking: "Cookie chemistry"
- King Arthur Baking: "The secret to fudgier cookies? It's all about the butter."
- King Arthur Baking: "Why are my cookies spreading?"

## Sugar Effects
Brown sugar contains molasses, which adds moisture and tends to support softer, chewier cookies.
Granulated sugar is often associated with more spread and a crisper texture.
Changing the balance between brown sugar and white sugar can noticeably change cookie texture.

Sources:
- King Arthur Baking: "Cookie chemistry"
- King Arthur Baking: "Soft Brown Sugar Cookies"

## Eggs and Egg Substitutes
Eggs help with structure, binding, and moisture. Removing them can make cookies more fragile or change how they spread and set.
Common egg substitutes in baking include flax egg, chia egg, applesauce, mashed banana, yogurt, and commercial egg replacers, but each substitute can change flavor or texture.
A common flax egg is made from 1 tablespoon ground flaxseed mixed with 3 tablespoons water for each egg.

Sources:
- King Arthur Baking: "No eggs? Here's your guide for substituting eggs"
- King Arthur Baking: "Our best recipes and tips for baking egg free"

## Dairy-Free Adjustments
Butter can often be replaced with plant-based butter or coconut oil, but flavor and spread may change.
If coconut oil is used, whether it is melted or solid can affect the final texture.

Sources:
- King Arthur Baking: "Cookie chemistry"
- King Arthur Baking: "The secret to fudgier cookies? It's all about the butter."

## Gluten-Free Adjustments
For many non-yeasted recipes, a 1-to-1 gluten-free flour blend can be used in place of all-purpose flour.
Even when the swap works, gluten-free cookies may be more delicate or somewhat different in texture than the original version.

Sources:
- King Arthur Baking: "Gluten-Free Baking"
- King Arthur Baking: "Gluten-Free Chocolate Chip Cookies"

## Leavening
Baking soda and baking powder are not the same. Baking powder already contains the components needed to create lift, while baking soda needs an acidic partner in the recipe.
In cookies, more leavening generally creates a puffier cookie, while lower leavening can reduce puffiness.
Baking soda also supports browning.

Sources:
- King Arthur Baking: "What's the difference between baking soda and baking powder substitutions"
- King Arthur Baking: "Drop cookies: beyond the fork"
- Sally's Baking Addiction: "Baking Powder vs. Baking Soda"

## Chilling Dough and Spread
Chilling cookie dough usually reduces spread and can improve flavor.
Refrigerated dough can develop a deeper flavor over time, and chilling is especially helpful when dough contains melted butter or otherwise feels warm.
If cookies spread too much, checking ingredient ratios, dough temperature, and flour measurement is a good starting point.

Sources:
- King Arthur Baking: "Chilling cookie dough: Does it make a difference?"
- King Arthur Baking: "Classic Chocolate Chip Cookies"
- King Arthur Baking: "Why are my cookies spreading?"

## Project Guidance for Substitutions
A substitution should not just replace the ingredient by name. It should also consider the ingredient's role in the recipe, such as flavor, moisture, structure, spread, or chewiness.
If the likely outcome is uncertain, the response should say so clearly instead of sounding overly confident.
"""
print("Source-backed cookie knowledge text created.")

Source-backed cookie knowledge text created.


**Section 4.3** — Save it as cookie_knowledge_base.txt

In [ ]:
kb_path = "/content/drive/MyDrive/ITAI2376_CookieAgent/data/knowledge_base/cookie_knowledge_base.txt"

with open(kb_path, "w", encoding="utf-8") as f:
    f.write(cookie_knowledge_text.strip())

print("Saved knowledge base to:", kb_path)

Saved knowledge base to: /content/drive/MyDrive/ITAI2376_CookieAgent/data/knowledge_base/cookie_knowledge_base.txt


**Section 4.4** — Preview the file

In [ ]:
with open(kb_path, "r", encoding="utf-8") as f:
    preview_text = f.read()

print(preview_text[:3000])

# Cookie Baking Knowledge Base

## Cookie Texture Basics
Chewier cookies usually come from higher moisture and a softer center. Brown sugar, melted butter, and shorter baking can all push cookies toward a chewier result.
Crispier cookies usually come from lower-moisture formulas, more granulated sugar, and longer baking.
Small changes in sugar, fat, and bake time can shift the same cookie recipe toward chewy, crispy, or cakey results.

Sources:
- King Arthur Baking: "Cookie chemistry"
- King Arthur Baking: "Cookie science: How to achieve your perfect chocolate chip cookie"

## Butter and Fat
Butter adds flavor and affects spread. Melted butter often increases spread and can lead to denser or fudgier centers, while softened butter is commonly used for creaming because it helps incorporate air into the dough.
If cookies spread too much, warm fat, excess liquid, or inaccurate ingredient ratios may be part of the problem.

Sources:
- King Arthur Baking: "Cookie chemistry"
- King Arthur Bak

# Section 5 — Building the RAG retriever

**Section 5.1** — Install packages

In [ ]:
%pip install -qU langchain langchain-huggingface langchain-chroma langchain-text-splitters sentence-transformers tiktoken

**Section 5.2** — Imports

In [ ]:
import pandas as pd
print("pandas version:", pd.__version__)

from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("All imports worked.")

pandas version: 2.2.2
All imports worked.


**Section 5.3** — Load your cleaned files

In [ ]:
# File paths
recipes_path = "/content/drive/MyDrive/ITAI2376_CookieAgent/cleaned_data/cookie_recipes_working_subset.csv"
kb_path = "/content/drive/MyDrive/ITAI2376_CookieAgent/data/knowledge_base/cookie_knowledge_base.txt"

# Load recipe subset
recipes_df = pd.read_csv(recipes_path)

# Load cookie knowledge base text
with open(kb_path, "r", encoding="utf-8") as f:
    kb_text = f.read()

print("Recipes shape:", recipes_df.shape)
print("Knowledge base length:", len(kb_text))
display(recipes_df.head(2))

Recipes shape: (3000, 15)
Knowledge base length: 4267


,id,name,description,ingredients,ingredients_raw,steps,servings,serving_size,tags,ingredients_raw_key,ingredients_text,ingredients_raw_text,steps_text,tags_text,retrieval_text
0,27239,Cranberry Cookies,I usually make these during the Christmas seas...,"['brown sugar', 'white sugar', 'vanilla', 'flo...","['1 cup butter', '2 cups brown sugar...",['In a very large bowl combine butter and brow...,1.0,1 (1981 g),"['30-minutes-or-less', 'time-to-make', 'course...",1 cup butter | 2 cups brown sugar | ...,brown sugar | white sugar | vanilla | flour | ...,1 cup butter | 2 cups brown sugar | ...,In a very large bowl combine butter and brown ...,30-minutes-or-less | time-to-make | course | p...,Recipe Name: Cranberry Cookies\nDescription: I...
1,306003,Chocolate Walnut Cookies,Good flavor of chocolate,"['icing sugar', 'cake flour', 'baking powder',...","['305 g butter', '470 g icing sugar'...",['Mix all together.'],5.0,1 (511 g),"['60-minutes-or-less', 'time-to-make', 'course...",305 g butter | 470 g icing sugar | 1...,icing sugar | cake flour | baking powder | bak...,305 g butter | 470 g icing sugar | 1...,Mix all together.,60-minutes-or-less | time-to-make | course | m...,Recipe Name: Chocolate Walnut Cookies\nDescrip...


**Section 5.4** — Turn recipe rows into LangChain documents

In [ ]:
recipe_docs = []

for _, row in recipes_df.iterrows():
    text = str(row.get("retrieval_text", "")).strip()

    if not text:
        continue

    recipe_docs.append(
        Document(
            page_content=text,
            metadata={
                "source_type": "recipe",
                "recipe_id": str(row.get("id", "")),
                "title": str(row.get("name", ""))
            }
        )
    )

print("Recipe documents created:", len(recipe_docs))
print("Sample recipe metadata:", recipe_docs[0].metadata if recipe_docs else "No docs")
print("\nSample recipe content:\n")
print(recipe_docs[0].page_content[:600] if recipe_docs else "No docs")

Recipe documents created: 3000
Sample recipe metadata: {'source_type': 'recipe', 'recipe_id': '27239', 'title': 'Cranberry Cookies'}

Sample recipe content:

Recipe Name: Cranberry Cookies
Description: I usually make these during the Christmas season. I entered this recipe into a Readers Digest Christmas cookie contest and received an honorable mention.
Ingredients: 1   cup    butter | 2   cups    brown sugar | 1   cup    white sugar | 2       eggs | 1/2  cup    water | 2   teaspoons    vanilla | 6   cups    flour | 2   teaspoons    salt | 2   teaspoons    baking soda | 1   package   washed whole cranberries | 1   cup   chopped walnuts or 1   cup    pecans
Steps: In a very large bowl combine butter and brown and white sugar; cream together Add eggs


**Section 5.5** — Turn the knowledge base into documents

In [ ]:
kb_docs = []

sections = [section.strip() for section in kb_text.split("## ") if section.strip()]

for section in sections:
    title = section.split("\n")[0].strip()

    kb_docs.append(
        Document(
            page_content=section,
            metadata={
                "source_type": "knowledge_base",
                "title": title
            }
        )
    )

print("Knowledge base documents created:", len(kb_docs))
print("Sample KB metadata:", kb_docs[0].metadata if kb_docs else "No docs")
print("\nSample KB content:\n")
print(kb_docs[0].page_content[:600] if kb_docs else "No docs")

Knowledge base documents created: 10
Sample KB metadata: {'source_type': 'knowledge_base', 'title': '# Cookie Baking Knowledge Base'}

Sample KB content:

# Cookie Baking Knowledge Base


**Section 5.6** — Combine everything

In [ ]:
all_docs = recipe_docs + kb_docs

print("Total original documents:", len(all_docs))

Total original documents: 3010


**Section 5.7** — Split long documents into smaller chunks

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

split_docs = text_splitter.split_documents(all_docs)

print("Total chunked documents:", len(split_docs))
print("Sample chunk metadata:", split_docs[0].metadata if split_docs else "No chunks")
print("\nSample chunk content:\n")
print(split_docs[0].page_content[:600] if split_docs else "No chunks")

Total chunked documents: 7135
Sample chunk metadata: {'source_type': 'recipe', 'recipe_id': '27239', 'title': 'Cranberry Cookies'}

Sample chunk content:

Recipe Name: Cranberry Cookies
Description: I usually make these during the Christmas season. I entered this recipe into a Readers Digest Christmas cookie contest and received an honorable mention.
Ingredients: 1   cup    butter | 2   cups    brown sugar | 1   cup    white sugar | 2       eggs | 1/2  cup    water | 2   teaspoons    vanilla | 6   cups    flour | 2   teaspoons    salt | 2   teaspoons    baking soda | 1   package   washed whole cranberries | 1   cup   chopped walnuts or 1   cup    pecans


**Section 5.8**— Create local embeddings

In [ ]:
# Local embedding model — no API key needed
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Local embedding model loaded.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Local embedding model loaded.


**Section 5.9** — Build the Chroma vector store

In [ ]:
persist_dir = "/content/drive/MyDrive/ITAI2376_CookieAgent/vectorstore/cookie_rag_chroma"

vectorstore = Chroma.from_documents(
    documents=split_docs,
    embedding=embeddings,
    persist_directory=persist_dir
)

print("Vector store created successfully.")
print("Saved at:", persist_dir)

Vector store created successfully.
Saved at: /content/drive/MyDrive/ITAI2376_CookieAgent/vectorstore/cookie_rag_chroma


**Section 5.10** — Create the retriever

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

print("Retriever is ready.")

Retriever is ready.


**Section 5.11** — Test retrieval with cookie questions

In [ ]:
test_questions = [
    "How can I make cookies chewier?",
    "What can I use instead of butter in cookies?",
    "What does brown sugar do in cookies?",
    "How do I scale a cookie recipe down?",
    "What happens if I use a flax egg in cookies?"
]

for question in test_questions:
    print("=" * 80)
    print("QUESTION:", question)

    results = retriever.invoke(question)

    for i, doc in enumerate(results, start=1):
        print(f"\nResult {i}")
        print("Metadata:", doc.metadata)
        print("Preview:", doc.page_content[:400])

QUESTION: How can I make cookies chewier?

Result 1
Metadata: {'title': 'Thick and Chewy Chocolate Chip Cookies', 'source_type': 'recipe', 'recipe_id': '410419'}
Preview: Recipe Name: Thick and Chewy Chocolate Chip Cookies
Description: These over-sized cookies are chewy and thick, like many of the chocolate chip cookies sold in gourmet shops and cookie stores. They rely on melted butter and an extra egg yolk to keep their texture soft. These cookies are best served warm from the oven but will retain their texture even when cooled. This may be the BEST chocolate chi

Result 2
Metadata: {'recipe_id': '410419', 'title': 'Thick and Chewy Chocolate Chip Cookies', 'source_type': 'recipe'}
Preview: Recipe Name: Thick and Chewy Chocolate Chip Cookies
Description: These over-sized cookies are chewy and thick, like many of the chocolate chip cookies sold in gourmet shops and cookie stores. They rely on melted butter and an extra egg yolk to keep their texture soft. These cookies are best served 

**Section 5.12** — Helper function for later

In [ ]:
def get_relevant_context(question, retriever, max_docs=4):
    docs = retriever.invoke(question)[:max_docs]

    context = "\n\n".join(
        [f"Source: {doc.metadata}\n{doc.page_content}" for doc in docs]
    )

    return context, docs

# Quick test
context_text, docs = get_relevant_context(
    "What happens if I replace butter with coconut oil in cookies?",
    retriever
)

print(context_text[:1500])

Source: {'recipe_id': '147787', 'source_type': 'recipe', 'title': 'Coconut Oatmeal Drop Cookies'}
Steps: Preheat the oven to 375°. | In a bowl, combine the first 4 ingredients; set aside. | In another larger bowl, cream the butter with the sugars for 2 minutes or until smooth. | Add in the egg and vanilla; beat well. | Add in the dry ingredients; mix thoroughly. | Stir in the oats and coconut. | Drop by rounded teaspoonfuls onto ungreased cookie sheets (leave room for expansion). | Bake for 12-14 minutes or until lightly golden. | Cool the cookies in the pan for 5 minutes; then transfer cookies to a wire rack to cool completely.
Tags: 60-minutes-or-less | time-to-make | course | main-ingredient | preparation | desserts | fruit | oven | cookies-and-brownies | nuts | dietary | coconut | equipment

Source: {'title': 'Coconut Oatmeal Drop Cookies', 'source_type': 'recipe', 'recipe_id': '147787'}
Steps: Preheat the oven to 375°. | In a bowl, combine the first 4 ingredients; set aside. | In 

# Section 6 — Defining the tools

**Tool 1 - Recipe Retrieval Tool:**

* take a user question
* use the retriever
* organize the results more clearly
* prioritize knowledge-base results first for “what happens if” or “what can I use instead of” questions
* still keep recipe examples if they are helpful

**Section 6.1** — Import the LangChain tool decorator

In [ ]:
from langchain_core.tools import tool

print("Tool decorator imported.")

Tool decorator imported.


**Section 6.2** — Create a helper function to organize retrieval results

In [ ]:
def organize_retrieval_results(question, docs):
    """
    Organize retrieved documents in a cleaner order depending on the question type.
    For substitution/explanation questions, prefer knowledge-base guidance first
    and only keep recipe examples that look relevant to substitutions.
    For recipe/example questions, prefer recipe documents first.
    """

    question_lower = question.lower()

    knowledge_docs = [doc for doc in docs if doc.metadata.get("source_type") == "knowledge_base"]
    recipe_docs = [doc for doc in docs if doc.metadata.get("source_type") == "recipe"]

    substitution_keywords = [
        "substitute", "substitutions", "replace", "replacement",
        "instead", "dairy-free", "dairy free", "vegan",
        "coconut oil", "oil", "plant-based", "flax", "egg substitute"
    ]

    # For substitution/explanation questions:
    # knowledge base first, and only recipe docs that mention helpful substitution terms
    if any(phrase in question_lower for phrase in [
        "instead of", "substitute", "replacement", "what happens", "why", "what does"
    ]):
        filtered_recipe_docs = [
            doc for doc in recipe_docs
            if any(keyword in doc.page_content.lower() for keyword in substitution_keywords)
        ]
        ordered_docs = knowledge_docs + filtered_recipe_docs

    # For recipe/example lookup questions:
    # recipe docs first, then knowledge base docs
    elif any(phrase in question_lower for phrase in [
        "recipe", "example", "show me", "find me"
    ]):
        ordered_docs = recipe_docs + knowledge_docs

    # Otherwise keep original order
    else:
        ordered_docs = docs

    return ordered_docs

**Section 6.3** — Define Tool 1 as a real LangChain tool

In [ ]:
@tool
def recipe_retrieval_tool(question: str) -> str:
    """
    Retrieve relevant cookie recipe and baking context for a user's question.
    Use this tool when the agent needs recipe examples, baking guidance,
    or background information about cookie ingredients, texture, or substitutions.
    """
    docs = retriever.invoke(question)

    if not docs:
        return "No relevant cookie recipe context was found."

    docs = organize_retrieval_results(question, docs)

    output_parts = []

    for i, doc in enumerate(docs[:4], start=1):
        source_type = doc.metadata.get("source_type", "unknown")
        title = doc.metadata.get("title", "Untitled")
        recipe_id = doc.metadata.get("recipe_id", "")

        header = f"Result {i} | Source Type: {source_type} | Title: {title}"
        if recipe_id:
            header += f" | Recipe ID: {recipe_id}"

        content = doc.page_content.strip()
        output_parts.append(f"{header}\n{content}")

    return "\n\n" + ("\n\n" + "=" * 80 + "\n\n").join(output_parts)

**Section 6.4** — Test Tool 1 with the butter question

In [ ]:
test_question = "What can I use instead of butter in cookies?"

tool_output = recipe_retrieval_tool.invoke({"question": test_question})

print(tool_output[:3000])



Result 1 | Source Type: knowledge_base | Title: Butter and Fat
Butter and Fat
Butter adds flavor and can help cookies spread. Melted butter usually encourages more spread than cold butter.
Softened butter is often used when creaming with sugar to add air and structure.
Oil can sometimes replace butter, but cookies may spread differently and lose some buttery flavor.
Coconut oil can work in some cookie recipes, but texture and flavor may change depending on whether it is used melted or solid.


Result 2 | Source Type: knowledge_base | Title: Butter and Fat
Butter and Fat
Butter adds flavor and can help cookies spread. Melted butter usually encourages more spread than cold butter.
Softened butter is often used when creaming with sugar to add air and structure.
Oil can sometimes replace butter, but cookies may spread differently and lose some buttery flavor.
Coconut oil can work in some cookie recipes, but texture and flavor may change depending on whether it is used melted or solid.




**Tool 2 - Ingredient Scaling Tool:**

* take recipe ingredient lines and serving information
* calculate a scale factor based on the original batch size and the desired batch size
* adjust ingredient amounts up or down
* return a scaled ingredient list in a clear readable format
* help the agent handle batch-size changes for cookie recipes

**Section 6.6** — Import regex and fractions

In [ ]:
import re
from fractions import Fraction

print("Regex and Fraction imported.")

Regex and Fraction imported.


**Section 6.7** — Create helper functions for parsing amounts

In [ ]:
def parse_fraction_string(amount_str):
    """
    Convert strings like '1', '1/2', '1 1/2' into float values.
    """
    amount_str = amount_str.strip()

    # Mixed number like "1 1/2"
    if " " in amount_str:
        parts = amount_str.split()
        if len(parts) == 2:
            whole = float(parts[0])
            frac = float(Fraction(parts[1]))
            return whole + frac

    # Simple fraction like "1/2"
    if "/" in amount_str:
        return float(Fraction(amount_str))

    # Regular number like "2" or "0.5"
    return float(amount_str)

**Section 6.8** — Create a basic ingredient scaling function

In [ ]:
from fractions import Fraction

def pretty_fraction(value, max_denominator=8):
    frac = Fraction(value).limit_denominator(max_denominator)

    if abs(float(frac) - float(value)) < 0.03:
        if frac.denominator == 1:
            return str(frac.numerator)
        return f"{frac.numerator}/{frac.denominator}"

    return f"{value:.2f}".rstrip("0").rstrip(".")


def format_baking_measure(amount, unit):
    unit = str(unit).strip().lower()

    if unit in ["tablespoon", "tablespoons"]:
        unit = "tbsp"
    elif unit in ["teaspoon", "teaspoons"]:
        unit = "tsp"
    elif unit in ["cups"]:
        unit = "cup"

    # convert large tablespoon amounts into cups when it looks nicer
    if unit == "tbsp" and amount >= 8:
        cup_amount = amount / 16
        quarter_cup = round(cup_amount * 4) / 4

        if abs(cup_amount - quarter_cup) < 0.03:
            amount_str = pretty_fraction(quarter_cup)
            cup_label = "cup" if abs(quarter_cup - 1) < 0.03 else "cups"
            return f"{amount_str} {cup_label}"

        whole_cups = int(amount // 16)
        remaining_tbsp = amount - (whole_cups * 16)

        parts = []
        if whole_cups > 0:
            cup_label = "cup" if whole_cups == 1 else "cups"
            parts.append(f"{whole_cups} {cup_label}")
        if remaining_tbsp > 0.05:
            parts.append(f"{pretty_fraction(remaining_tbsp)} tbsp")

        return " + ".join(parts)

    amount_str = pretty_fraction(amount)

    if unit == "cup":
        unit_label = "cup" if amount_str == "1" else "cups"
        return f"{amount_str} {unit_label}"

    if unit == "tbsp":
        return f"{amount_str} tbsp"

    if unit == "tsp":
        return f"{amount_str} tsp"

    return f"{amount_str} {unit}".strip()


def scale_ingredient_line(line, scale_factor):
    """
    Scale the first number or fraction found in an ingredient line.
    Example:
    '1 1/2 cups flour' with factor 2 -> '3 cups flour'
    """

    line = str(line).strip()

    match = re.match(r"^\s*(\d+\s+\d+/\d+|\d+/\d+|\d+(\.\d+)?)", line)
    if not match:
        return line

    original_amount_str = match.group(1)

    try:
        original_amount = parse_fraction_string(original_amount_str)
        scaled_amount = original_amount * scale_factor

        # try to detect unit right after the amount
        remainder = line[len(match.group(0)):].strip()
        unit_match = re.match(r"^(cups?|tablespoons?|teaspoons?|tbsp|tsp)\b", remainder, flags=re.IGNORECASE)

        if unit_match:
            original_unit = unit_match.group(1)
            rest_after_unit = remainder[len(unit_match.group(0)):].strip()
            pretty_measure = format_baking_measure(scaled_amount, original_unit)
            scaled_line = f"{pretty_measure} {rest_after_unit}".strip()
        else:
            # fallback if no common unit is found
            if scaled_amount.is_integer():
                scaled_amount_str = str(int(scaled_amount))
            else:
                scaled_amount_str = pretty_fraction(scaled_amount)

            scaled_line = line.replace(original_amount_str, scaled_amount_str, 1)

        # clean up singular/plural
        scaled_line = re.sub(r"^1\s+cups\b", "1 cup", scaled_line)
        scaled_line = re.sub(r"^1\s+tablespoons\b", "1 tablespoon", scaled_line)
        scaled_line = re.sub(r"^1\s+teaspoons\b", "1 teaspoon", scaled_line)
        scaled_line = re.sub(r"^1\s+eggs\b", "1 egg", scaled_line)

        return scaled_line

    except:
        return line

**Section 6.9** — Define Tool 2 as a LangChain tool

In [ ]:
@tool
def ingredient_scaling_tool(recipe_text: str, original_servings: float, desired_servings: float) -> str:
    """
    Scale ingredient amounts in a cookie recipe based on original and desired servings.
    Use this tool when the user wants to increase or decrease a cookie recipe batch size.
    """

    if original_servings <= 0 or desired_servings <= 0:
        return "Original servings and desired servings must both be greater than zero."

    scale_factor = desired_servings / original_servings

    lines = recipe_text.split("|")
    scaled_lines = [scale_ingredient_line(line.strip(), scale_factor) for line in lines if line.strip()]

    output = [
        f"Original servings: {original_servings}",
        f"Desired servings: {desired_servings}",
        f"Scale factor: {scale_factor:.2f}",
        "",
        "Scaled ingredients:"
    ]

    output.extend([f"- {line}" for line in scaled_lines])

    return "\n".join(output)

**Section 6.10** — Test Tool 2 with a small manual example

In [ ]:
sample_recipe_text = "1 cup butter | 1/2 cup brown sugar | 2 cups flour | 1 egg | 1 tsp vanilla"

scaled_output = ingredient_scaling_tool.invoke({
    "recipe_text": sample_recipe_text,
    "original_servings": 24,
    "desired_servings": 12
})

print(scaled_output)

Original servings: 24.0
Desired servings: 12.0
Scale factor: 0.50

Scaled ingredients:
- 1/2 cups butter
- 1/4 cups brown sugar
- 1 cup flour
- 1/2 egg
- 1/2 tsp vanilla


**Tool 3 - Substitution Guidance Tool:**

* take a user question about a missing ingredient or dietary adjustment
* identify the most likely ingredient or baking need involved
* suggest a practical cookie-friendly substitution
* explain likely effects on texture, spread, flavor, or structure
* use the knowledge base and retrieval system to support substitution guidance

**Section 6.12** — Create substitution guidance rules

In [ ]:
substitution_rules = {
    "butter": {
        "suggestions": [
            "coconut oil",
            "plant-based butter"
        ],
        "effects": [
            "Cookies may spread differently.",
            "Flavor may be less buttery.",
            "Texture may change depending on whether the replacement is solid or melted."
        ]
    },
    "egg": {
        "suggestions": [
            "flax egg (1 tablespoon ground flaxseed + 3 tablespoons water)",
            "applesauce",
            "mashed banana"
        ],
        "effects": [
            "Cookies may be more fragile.",
            "Texture and spread may change.",
            "Banana can add flavor, while flax egg is usually more neutral."
        ]
    },
    "milk": {
        "suggestions": [
            "almond milk",
            "oat milk",
            "soy milk"
        ],
        "effects": [
            "Most cookie recipes use little milk, so the change is often small.",
            "Flavor and moisture may shift slightly depending on the substitute."
        ]
    },
    "brown sugar": {
        "suggestions": [
            "white sugar plus a small amount of molasses",
            "all white sugar if necessary"
        ],
        "effects": [
            "Replacing brown sugar can reduce chewiness and moisture.",
            "Cookies may be crisper and less soft."
        ]
    },
    "white sugar": {
        "suggestions": [
            "brown sugar"
        ],
        "effects": [
            "Using brown sugar instead can make cookies softer and chewier.",
            "Flavor may be deeper because of the molasses content."
        ]
    },
    "all-purpose flour": {
        "suggestions": [
            "1-to-1 gluten-free flour blend",
            "whole wheat flour"
        ],
        "effects": [
            "Gluten-free flour may make cookies more delicate or crumbly.",
            "Whole wheat flour may make cookies denser and slightly nuttier."
        ]
    }
}

print("Substitution rules created.")

Substitution rules created.


**Section 6.13** — Create a helper function to identify the substitution request

In [ ]:
def identify_substitution_target(question):
    question_lower = question.lower()

    keyword_map = {
        "butter": ["butter", "dairy-free", "dairy free"],
        "egg": ["egg", "egg-free", "egg free", "vegan"],
        "milk": ["milk"],
        "brown sugar": ["brown sugar"],
        "white sugar": ["white sugar", "granulated sugar"],
        "all-purpose flour": ["all-purpose flour", "all purpose flour", "flour", "gluten-free", "gluten free"]
    }

    for target, keywords in keyword_map.items():
        if any(keyword in question_lower for keyword in keywords):
            return target

    return None

# Quick check
print(identify_substitution_target("What can I use instead of butter in cookies?"))
print(identify_substitution_target("How do I make this cookie recipe egg-free?"))

butter
egg


**Section 6.14** — Define Tool 3 as a real LangChain tool

In [ ]:
@tool
def substitution_guidance_tool(question: str) -> str:
    """
    Suggest cookie-friendly ingredient substitutions and explain likely effects
    on texture, spread, flavor, or structure.
    Use this tool when the user asks about replacing an ingredient or adjusting
    a cookie recipe for dietary needs.
    """

    target = identify_substitution_target(question)

    # Pull retrieved context
    docs = retriever.invoke(question)

    recipe_docs = [doc for doc in docs if doc.metadata.get("source_type") == "recipe"]
    retrieved_kb_docs = [doc for doc in docs if doc.metadata.get("source_type") == "knowledge_base"]

    # Map targets to preferred knowledge-base titles
    preferred_kb_titles_map = {
        "butter": ["Butter and Fat", "Dairy-Free Adjustments"],
        "egg": ["Eggs"],
        "brown sugar": ["Sugar Effects"],
        "white sugar": ["Sugar Effects"],
        "all-purpose flour": ["Gluten-Free Adjustments", "Flour Effects"],
        "milk": ["Dairy-Free Adjustments"]
    }

    preferred_titles = preferred_kb_titles_map.get(target, [])

    # First, directly pull matching KB docs from the full kb_docs list
    direct_preferred_kb_docs = [
        doc for doc in kb_docs
        if doc.metadata.get("title", "") in preferred_titles
    ]

    # Then keep any other retrieved KB docs that were found
    other_retrieved_kb_docs = [
        doc for doc in retrieved_kb_docs
        if doc.metadata.get("title", "") not in preferred_titles
    ]

    substitution_keywords = [
        "substitute", "substitution", "replace", "replacement",
        "instead", "dairy-free", "dairy free", "vegan",
        "coconut oil", "plant-based", "flax", "egg-free", "egg free",
        "gluten-free", "gluten free", "applesauce", "banana"
    ]

    # Keep only recipe docs that seem useful for substitution guidance
    filtered_recipe_docs = [
        doc for doc in recipe_docs
        if any(keyword in doc.page_content.lower() for keyword in substitution_keywords)
    ]

    # Combine in the order we want
    selected_docs = direct_preferred_kb_docs + other_retrieved_kb_docs + filtered_recipe_docs

    # Remove duplicates by title + source_type
    seen = set()
    unique_docs = []
    for doc in selected_docs:
        key = (doc.metadata.get("source_type", ""), doc.metadata.get("title", ""))
        if key not in seen:
            seen.add(key)
            unique_docs.append(doc)

    selected_docs = unique_docs[:3]

    context_preview = "\n\n".join(
        [f"Source: {doc.metadata}\n{doc.page_content[:400]}" for doc in selected_docs]
    )

    if target and target in substitution_rules:
        rule = substitution_rules[target]

        response_parts = [
            f"Question: {question}",
            f"Identified target: {target}",
            "",
            "Suggested substitutions:"
        ]

        response_parts.extend([f"- {item}" for item in rule["suggestions"]])

        response_parts.append("")
        response_parts.append("Likely effects:")
        response_parts.extend([f"- {item}" for item in rule["effects"]])

        response_parts.append("")
        response_parts.append("Relevant retrieved context:")
        response_parts.append(context_preview if context_preview else "No additional supporting context found.")

        return "\n".join(response_parts)

    else:
        return (
            f"Question: {question}\n\n"
            "I could not confidently match this to one of the built-in substitution rules yet.\n\n"
            "Relevant retrieved context:\n"
            f"{context_preview if context_preview else 'No additional supporting context found.'}"
        )

**Section 6.15** — Test Tool 3 with a butter substitution question

In [ ]:
butter_sub_output = substitution_guidance_tool.invoke({
    "question": "What can I use instead of butter in cookies?"
})

print(butter_sub_output[:3000])

Question: What can I use instead of butter in cookies?
Identified target: butter

Suggested substitutions:
- coconut oil
- plant-based butter

Likely effects:
- Cookies may spread differently.
- Flavor may be less buttery.
- Texture may change depending on whether the replacement is solid or melted.

Relevant retrieved context:
Source: {'source_type': 'knowledge_base', 'title': 'Butter and Fat'}
Butter and Fat
Butter adds flavor and affects spread. Melted butter often increases spread and can lead to denser or fudgier centers, while softened butter is commonly used for creaming because it helps incorporate air into the dough.
If cookies spread too much, warm fat, excess liquid, or inaccurate ingredient ratios may be part of the problem.

Sources:
- King Arthur Baking: "Cookie chemistry"
-

Source: {'source_type': 'knowledge_base', 'title': 'Dairy-Free Adjustments'}
Dairy-Free Adjustments
Butter can often be replaced with plant-based butter or coconut oil, but flavor and spread may chan

**Tool 4 - Recipe Adaptation Agent:**

* take a specific cookie recipe and an adaptation request
* identify which ingredient or dietary adjustment is being requested
* rewrite the ingredient list using a practical substitution
* preserve the rest of the recipe as closely as possible
* explain likely effects on texture, flavor, spread, or structure

**Section 6.17** — Create helper functions for recipe adaptation

In [ ]:
def split_recipe_ingredients(recipe_text):
    """
    Split a pasted ingredient list into separate lines.
    Supports either newline-separated text or | separated text.
    """
    if "|" in recipe_text:
        parts = recipe_text.split("|")
    else:
        parts = recipe_text.split("\n")

    cleaned = []
    for part in parts:
        line = str(part).strip()
        line = line.replace("▢", "").strip()
        if line:
            cleaned.append(line)

    return cleaned


def extract_leading_amount(line):
    """
    Extract the first number/fraction/mixed number from the start of a line.
    Returns a float if found, otherwise None.
    Examples:
    '2 large eggs' -> 2.0
    '1/2 cup milk' -> 0.5
    '1 1/2 cups flour' -> 1.5
    """
    line = str(line).strip()
    match = re.match(r"^\s*(\d+\s+\d+/\d+|\d+/\d+|\d+(\.\d+)?)|", line)

    if not match:
        return None

    amount_str = match.group(1)

    try:
        return parse_fraction_string(amount_str)
    except:
        return None


def adapt_ingredient_lines(ingredient_lines, target):
    """
    Replace ingredient lines for the requested adaptation target.
    Returns adapted lines plus a list of what changed.
    """
    adapted_lines = []
    changes_made = []

    for line in ingredient_lines:
        lower_line = line.lower()

        # Butter replacement
        if target == "butter" and "butter" in lower_line:
            butter_amount = extract_leading_amount(line)

            if butter_amount is None:
                adapted_lines.append(
                    line + "  →  substitute with plant-based butter or coconut oil"
                )
            else:
                butter_text = format_baking_measure(butter_amount, "cup")

                adapted_lines.append(
                    f"{line}  →  substitute with {butter_text} plant-based butter "
                    f"or {butter_text} coconut oil"
                )

            changes_made.append(
                "Replaced butter with plant-based butter or coconut oil while noting likely flavor and texture changes."
            )

        # Egg replacement
        elif target == "egg" and re.search(r"\begg(s)?\b", lower_line):
            egg_count = extract_leading_amount(line)

            if egg_count is None:
                adapted_lines.append(line + "  →  substitute with flax egg, applesauce, or mashed banana")
            else:
                flax_tbsp = egg_count
                water_tbsp = egg_count * 3
                applesauce_cups = egg_count * 0.25
                banana_cups = egg_count * 0.25

                flax_text = pretty_fraction(flax_tbsp)
                water_text = format_baking_measure(water_tbsp, "tbsp")
                applesauce_text = format_baking_measure(applesauce_cups, "cup")
                banana_text = format_baking_measure(banana_cups, "cup")

                adapted_lines.append(
                    line +
                    f"  →  substitute with {flax_text} flax egg(s) "
                    f"({flax_text} tbsp ground flaxseed + {water_text} water), "
                    f"or {applesauce_text} applesauce, or {banana_text} mashed banana"
                )

            changes_made.append("Replaced egg guidance with proportional flax egg, applesauce, or mashed banana.")

        # Brown sugar replacement
        elif target == "brown sugar" and "brown sugar" in lower_line:
            adapted_lines.append(line + "  →  substitute with white sugar plus a small amount of molasses, or all white sugar if needed")
            changes_made.append("Replaced brown sugar guidance with white sugar + molasses or white sugar only.")

        # White sugar replacement
        elif target == "white sugar" and ("white sugar" in lower_line or "granulated sugar" in lower_line):
            adapted_lines.append(line + "  →  substitute with brown sugar")
            changes_made.append("Replaced white sugar guidance with brown sugar.")

        # Flour replacement
        elif target == "all-purpose flour" and ("all-purpose flour" in lower_line or "all purpose flour" in lower_line or re.search(r"\bflour\b", lower_line)):
            adapted_lines.append(line + "  →  substitute with 1-to-1 gluten-free flour blend or whole wheat flour depending on the goal")
            changes_made.append("Replaced flour guidance with gluten-free blend or whole wheat flour option.")

        # Milk replacement
        elif target == "milk" and "milk" in lower_line:
            adapted_lines.append(line + "  →  substitute with almond milk, oat milk, or soy milk")
            changes_made.append("Replaced milk guidance with a plant-based milk.")

        else:
            adapted_lines.append(line)

    return adapted_lines, changes_made

**Section 6.18** — Define Tool 4 as a real LangChain tool

In [ ]:
@tool
def recipe_adaptation_tool(question: str, recipe_text: str) -> str:
    """
    Adapt a specific cookie recipe based on a substitution or dietary request.
    Use this tool when the user provides a cookie recipe and wants that exact recipe adapted.
    """

    target = identify_substitution_target(question)
    ingredient_lines = split_recipe_ingredients(recipe_text)

    if not ingredient_lines:
        return "I could not read the recipe ingredients clearly. Please provide the ingredient list in text form."

    docs = retriever.invoke(question)
    knowledge_docs = [doc for doc in docs if doc.metadata.get("source_type") == "knowledge_base"]

    preferred_kb_titles_map = {
        "butter": ["Butter and Fat", "Dairy-Free Adjustments"],
        "egg": ["Eggs"],
        "brown sugar": ["Sugar Effects"],
        "white sugar": ["Sugar Effects"],
        "all-purpose flour": ["Gluten-Free Adjustments", "Flour Effects"],
        "milk": ["Dairy-Free Adjustments"]
    }

    preferred_titles = preferred_kb_titles_map.get(target, [])
    direct_preferred_kb_docs = [
        doc for doc in kb_docs
        if doc.metadata.get("title", "") in preferred_titles
    ]

    selected_kb_docs = (direct_preferred_kb_docs + knowledge_docs)[:2]

    if target and target in substitution_rules:
        adapted_lines, changes_made = adapt_ingredient_lines(ingredient_lines, target)
        rule = substitution_rules[target]

        response_parts = [
            f"Adaptation request: {question}",
            f"Identified target: {target}",
            "",
            "Adapted ingredient list:"
        ]

        response_parts.extend([f"- {line}" for line in adapted_lines])

        response_parts.append("")
        response_parts.append("Key substitution suggestions:")
        response_parts.extend([f"- {item}" for item in rule["suggestions"]])

        response_parts.append("")
        response_parts.append("Likely effects:")
        response_parts.extend([f"- {item}" for item in rule["effects"]])

        if changes_made:
            response_parts.append("")
            response_parts.append("Changes applied:")
            for change in sorted(set(changes_made)):
                response_parts.append(f"- {change}")

        if selected_kb_docs:
            response_parts.append("")
            response_parts.append("Supporting baking guidance:")
            for doc in selected_kb_docs:
                response_parts.append(f"- {doc.metadata.get('title', 'Untitled')}: {doc.page_content[:250]}")

        return "\n".join(response_parts)

    return (
        "I could not confidently identify the requested adaptation target from that question yet. "
        "Try asking more directly, such as 'Make this cookie recipe egg-free' or 'Adapt this recipe to replace butter.'"
    )

**Section 6.19** — Test Tool 4 with an egg-free adaptation

In [ ]:
recipe_text_example = """
1 cup salted butter softened
1 cup granulated sugar
1 cup light brown sugar packed
2 teaspoons pure vanilla extract
2 large eggs
3 cups all-purpose flour
1 teaspoon baking soda
1/2 teaspoon baking powder
1 teaspoon sea salt
2 cups chocolate chips
"""

tool4_output = recipe_adaptation_tool.invoke({
    "question": "Make this cookie recipe egg-free",
    "recipe_text": recipe_text_example
})

print(tool4_output[:4000])

Adaptation request: Make this cookie recipe egg-free
Identified target: egg

Adapted ingredient list:
- 1 cup salted butter softened
- 1 cup granulated sugar
- 1 cup light brown sugar packed
- 2 teaspoons pure vanilla extract
- 2 large eggs  →  substitute with 2 flax egg(s) (2 tbsp ground flaxseed + 6 tbsp water), or 1/2 cups applesauce, or 1/2 cups mashed banana
- 3 cups all-purpose flour
- 1 teaspoon baking soda
- 1/2 teaspoon baking powder
- 1 teaspoon sea salt
- 2 cups chocolate chips

Key substitution suggestions:
- flax egg (1 tablespoon ground flaxseed + 3 tablespoons water)
- applesauce
- mashed banana

Likely effects:
- Cookies may be more fragile.
- Texture and spread may change.
- Banana can add flavor, while flax egg is usually more neutral.

Changes applied:
- Replaced egg guidance with proportional flax egg, applesauce, or mashed banana.


**Section 6.20** — Test Tool 4 with a butter substitution

In [ ]:
tool4_butter_output = recipe_adaptation_tool.invoke({
    "question": "Adapt this cookie recipe to replace butter",
    "recipe_text": recipe_text_example
})

print(tool4_butter_output[:4000])

Adaptation request: Adapt this cookie recipe to replace butter
Identified target: butter

Adapted ingredient list:
- 1 cup salted butter softened  →  substitute with 1 cup plant-based butter or 1 cup coconut oil
- 1 cup granulated sugar
- 1 cup light brown sugar packed
- 2 teaspoons pure vanilla extract
- 2 large eggs
- 3 cups all-purpose flour
- 1 teaspoon baking soda
- 1/2 teaspoon baking powder
- 1 teaspoon sea salt
- 2 cups chocolate chips

Key substitution suggestions:
- coconut oil
- plant-based butter

Likely effects:
- Cookies may spread differently.
- Flavor may be less buttery.
- Texture may change depending on whether the replacement is solid or melted.

Changes applied:
- Replaced butter with plant-based butter or coconut oil while noting likely flavor and texture changes.

Supporting baking guidance:
- Butter and Fat: Butter and Fat
Butter adds flavor and affects spread. Melted butter often increases spread and can lead to denser or fudgier centers, while softened butter i

# Section 7 — Building the agent

**Section 7.1** — Install the OpenAI LangChain package

In [ ]:
%pip install -qU langchain-openai

**Section 7.2**— Import the agent builder

In [ ]:
from langchain.agents import create_agent
import os
import getpass

print("Agent builder imported.")

Agent builder imported.


**Section 7.3** — Set the OpenAI API key privately in Colab

In [ ]:
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OPENAI_API_KEY: ")

print("OpenAI API key is set for this Colab session.")

OpenAI API key is set for this Colab session.


**Section 7.4** — Create the system prompt

In [ ]:
SYSTEM_PROMPT = """
You are a beginner-friendly cookie recipe adaptation agent.

Your job is to help users adapt cookie recipes based on:
1. batch size changes
2. ingredient substitutions
3. dietary adjustments

You have access to four tools:
- recipe_retrieval_tool: use this when you need recipe examples, cookie baking guidance, or background information
- ingredient_scaling_tool: use this when the user wants to increase or decrease a cookie recipe batch size
- substitution_guidance_tool: use this when the user wants to replace an ingredient or make a dietary adjustment
- recipe_adaptation_tool: use this when the user provides a specific cookie recipe and wants that exact recipe adapted

Behavior rules:
- Always stay focused on cookie recipes.
- If the user asks about a non-cookie recipe, clearly say that your current scope is cookie recipes only.
- Prefer adapting the user’s current cookie recipe rather than suggesting a completely different recipe.
- For substitution questions, give direct substitutions first, then explain likely effects on texture, spread, flavor, or structure.
- If the user provides a specific recipe and asks for a substitution or dietary change, use the recipe_adaptation_tool.
- Do not list full alternative recipes unless the user specifically asks for recipe examples.
- For scaling questions, use the ingredient_scaling_tool when the user provides the recipe text and serving information.
- If the user asks to scale a recipe but does not provide enough information, ask for the missing ingredient list or serving numbers.
- Keep answers clear, concise, and beginner-friendly.
- Be honest when a substitution may change the final result.
- When a quantity becomes fractional, it is okay to explain practical baking handling, such as whisking an egg and using half.
- When presenting ingredient amounts, use natural baking measurements. Prefer clean fractions and convert large tablespoon amounts into cups when that is more readable.
"""
print("Updated system prompt created.")

Updated system prompt created.


**Section 7.5** — Bundling the tools

In [ ]:
tools = [
    recipe_retrieval_tool,
    ingredient_scaling_tool,
    substitution_guidance_tool,
    recipe_adaptation_tool
]

print("Tools ready:", [tool.name for tool in tools])

Tools ready: ['recipe_retrieval_tool', 'ingredient_scaling_tool', 'substitution_guidance_tool', 'recipe_adaptation_tool']


**Section 7.6** — Creating the LangChain agent

In [ ]:
agent = create_agent(
    model="openai:gpt-4o-mini",
    tools=tools,
    system_prompt=SYSTEM_PROMPT
)

print("Agent created successfully.")

Agent created successfully.


**Section 7.7** — Test the agent with a substitution question

In [ ]:
result = agent.invoke({
    "messages": [
        {"role": "user", "content": "What can I use instead of butter in cookies?"}
    ]
})

print(result["messages"][-1].content)

You can use the following substitutes for butter in cookies:

- **Coconut Oil**: Use it in a 1:1 ratio. If it's melted, it may cause cookies to spread more.
- **Plant-Based Butter**: Also a 1:1 substitution.

### Likely Effects:
- The cookies may spread differently compared to when using butter.
- The flavor may not be as buttery, which can affect the overall taste.
- The texture can vary; solid versus melted substitutes will impact the final result.

Using these substitutes will still yield delicious cookies, but it's good to be aware of the potential differences!


**Section 7.10** — Test the agent with a scaling question

In [ ]:
scaling_prompt = """
Scale this cookie recipe from 24 servings to 12 servings.

Recipe ingredients:
1 cup butter | 1/2 cup brown sugar | 2 cups flour | 1 egg | 1 tsp vanilla
"""

result = agent.invoke({
    "messages": [
        {"role": "user", "content": scaling_prompt}
    ]
})

print(result["messages"][-1].content)

Here are the scaled ingredients for your cookie recipe, adjusted from 24 servings to 12 servings:

- **1/2 cup** butter
- **1/4 cup** brown sugar
- **1 cup** flour
- **1/2 egg** (you can whisk an egg and use half)
- **1/2 tsp** vanilla

Enjoy baking your cookies!


**Section 7.11** — Make a helper function for later testing

In [ ]:
def run_cookie_agent(user_question):
    result = agent.invoke({
        "messages": [
            {"role": "user", "content": user_question}
        ]
    })
    return result["messages"][-1].content

# Quick test
print(run_cookie_agent("What does brown sugar do in cookies?"))

Brown sugar plays several important roles in cookie recipes:

1. **Moisture**: Brown sugar contains more moisture than white sugar due to the presence of molasses. This helps to keep cookies chewy and soft.

2. **Flavor**: The molasses in brown sugar adds a deeper, richer flavor, with hints of caramel and toffee that enhances the overall taste of the cookies.

3. **Color**: Cookies made with brown sugar tend to have a darker color, giving them a more appealing appearance.

4. **Spread**: Brown sugar can affect the spread of cookies. It usually results in a more spread-out, chewier cookie compared to white sugar, which can lead to a softer texture.

Overall, substituting or omitting brown sugar will influence the cookie's texture, flavor, and appearance. If you have specific substitutions in mind, let me know!


**Section 7.15**

In [ ]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": """Make this cookie recipe egg-free:

1 cup salted butter softened
1 cup granulated sugar
1 cup light brown sugar packed
2 teaspoons pure vanilla extract
2 large eggs
3 cups all-purpose flour
1 teaspoon baking soda
1/2 teaspoon baking powder
1 teaspoon sea salt
2 cups chocolate chips"""
        }
    ]
})

print(result["messages"][-1].content)

Here’s your adapted cookie recipe made egg-free:

- 1 cup salted butter, softened
- 1 cup granulated sugar
- 1 cup light brown sugar, packed
- 2 teaspoons pure vanilla extract
- **Egg Substitute**: Use one of the following options:
  - **Flax Egg**: Mix 2 tablespoons of ground flaxseed with 6 tablespoons of water. Let it sit for about 5 minutes until it thickens.
  - **Applesauce**: Use 1/2 cup applesauce.
  - **Mashed Banana**: Use 1/2 cup mashed banana.
- 3 cups all-purpose flour
- 1 teaspoon baking soda
- 1/2 teaspoon baking powder
- 1 teaspoon sea salt
- 2 cups chocolate chips

### Key Considerations:
- If using a flax egg, the flavor will remain more neutral, while using mashed banana may give a distinct banana flavor.
- The cookies may be a bit more fragile and their texture and spread could change compared to using eggs.

Feel free to choose the substitute that best fits your preference!


# Section 8 — Testing real-world scenarios

These test scenarios are designed to show the three main abilities of the cookie recipe adaptation agent:

1. ingredient substitution
2. dietary adjustment
3. batch size scaling

Additional scenarios can also help test general cookie guidance and scope control.

**Section 8.1** — Scenario 1: Butter substitution

In [ ]:
scenario_1 = "What can I use instead of butter in cookies?"

print("Scenario 1 — Butter substitution")
print(run_cookie_agent(scenario_1))

Scenario 1 — Butter substitution
You can substitute butter in cookie recipes with:

1. **Coconut Oil**: Use in the same quantity as butter. If it's solid, it will create a slightly different texture than butter; if melted, it may lead to more spread.
  
2. **Plant-Based Butter**: This can be swapped in directly for butter in equal amounts, but the flavor will be less buttery.

### Likely Effects:
- **Spread**: Cookies may spread differently depending on the fat used.
- **Flavor**: The cookies might taste less rich without butter.
- **Texture**: Depending on whether the replacement is solid or melted, the texture could be affected; coconut oil, for instance, can create a denser or fudgier center if melted.

If you have a particular cookie recipe in mind, I can help you adapt that recipe with the substitutions!


**Section 8.2** — Scenario 2: Egg-free adjustment

In [ ]:
scenario_2 = "How do I make a cookie recipe egg-free?"

print("Scenario 2 — Egg-free adjustment")
print(run_cookie_agent(scenario_2))

Scenario 2 — Egg-free adjustment
To make a cookie recipe egg-free, here are some common substitutions you can use:

1. **Unsweetened Applesauce**: Use 1/4 cup of applesauce for each egg. This adds moisture and a bit of sweetness.

2. **Flaxseed Meal**: Mix 1 tablespoon of flaxseed meal with 2.5 tablespoons of water. Let it sit for about 5 minutes to thicken and then use it as a replacement for one egg.

3. **Chia Seeds**: Similar to flaxseed, you can use 1 tablespoon of chia seeds mixed with 2.5 tablespoons of water, allowed to sit until it gels.

4. **Mashed Banana**: Use 1/4 cup of mashed banana for each egg. It adds a slight banana flavor and moisture.

5. **Silken Tofu**: Blend 1/4 cup of silken tofu until smooth for each egg. This will add density and moisture.

Each of these substitutes can affect the texture and flavor of the cookies. For example, applesauce and bananas can make cookies softer and chewier, while flax and chia seeds may add a slight nuttiness. If you want to adap

**Section 8.3** — Scenario 3: Batch scaling

In [ ]:
scenario_3 = """
Scale this cookie recipe from 24 servings to 12 servings.

Recipe ingredients:
1 cup butter | 1/2 cup brown sugar | 2 cups flour | 1 egg | 1 tsp vanilla
"""

print("Scenario 3 — Batch scaling")
print(run_cookie_agent(scenario_3))

Scenario 3 — Batch scaling
To scale your cookie recipe down from 24 servings to 12 servings, here are the adjusted ingredient amounts:

- **1/2 cup butter**
- **1/4 cup brown sugar**
- **1 cup flour**
- **1/2 egg** (you can whisk an egg and use half)
- **1/2 tsp vanilla**

Now you have the perfect scaled-down recipe! Happy baking!


**Section 8.4** — Scenario 4: Texture guidance

In [ ]:
scenario_4 = "How can I make cookies chewier?"

print("Scenario 4 — Texture guidance")
print(run_cookie_agent(scenario_4))

Scenario 4 — Texture guidance
To make cookies chewier, consider these tips:

1. **Use Brown Sugar**: Replacing some or all of the granulated sugar with brown sugar can enhance chewiness because brown sugar contains moisture.

2. **Add an Extra Egg Yolk**: This increases the fat content, leading to a chewier texture.

3. **Decrease Baking Time**: Slightly underbaking your cookies will keep them soft and chewy.

4. **Chill the Dough**: Allow your cookie dough to chill in the refrigerator for at least an hour before baking. This helps the cookies maintain their shape and encourages a chewy texture.

5. **Use Bread Flour**: This flour has more protein than all-purpose flour and contributes to a chewier texture.

6. **Add Cornstarch**: A small amount of cornstarch in the dough can help create a softer, chewier cookie.

Feel free to ask if you want specific advice for a cookie recipe you have!


**Section 8.5** — Scenario 5: Scope control

In [ ]:
scenario_5 = "Can you help me adapt a brownie recipe to be dairy-free?"

print("Scenario 5 — Scope control")
print(run_cookie_agent(scenario_5))

Scenario 5 — Scope control
Currently, I can only assist with cookie recipes. If you have a specific cookie recipe you'd like to adapt to be dairy-free, please provide it, and I'll be happy to help!


**Section 8.6** — Run all scenarios in one place

In [ ]:
scenario_dict = {
    "Scenario 1 — Butter substitution": scenario_1,
    "Scenario 2 — Egg-free adjustment": scenario_2,
    "Scenario 3 — Batch scaling": scenario_3,
    "Scenario 4 — Texture guidance": scenario_4,
    "Scenario 5 — Scope control": scenario_5
}

for title, prompt in scenario_dict.items():
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)
    print(run_cookie_agent(prompt))


Scenario 1 — Butter substitution
You can use the following as substitutes for butter in cookies:

1. **Coconut Oil**: Can be used in a 1:1 ratio. Use melted coconut oil for a more similar texture to melted butter, or solid for a thicker texture.
2. **Plant-Based Butter**: Also a 1:1 substitute, designed to mimic the flavor and texture of regular butter.

### Likely Effects:
- Using **coconut oil** may lead to a different spread compared to butter, and the flavor will be less buttery, giving cookies a slightly different taste. The texture can also vary depending on whether you use it melted or solid.
- **Plant-based butter** generally provides a taste closer to traditional butter but can still affect how the cookies spread.

When replacing butter, keep in mind that the texture and spread of the cookies can change, especially with melted versus solid substitutes.

Scenario 2 — Egg-free adjustment
To make a cookie recipe egg-free, you can use substitutes such as:

1. **Applesauce**: Use 

**Section 8.7** — Saving the recommended final demo scenarios

In [ ]:
recommended_demo_scenarios = {
    "Demo Scenario 1": "What can I use instead of butter in cookies?",
    "Demo Scenario 2": "How do I make a cookie recipe egg-free?",
    "Demo Scenario 3": """
Scale this cookie recipe from 24 servings to 12 servings.

Recipe ingredients:
1 cup butter | 1/2 cup brown sugar | 2 cups flour | 1 egg | 1 tsp vanilla
"""
}

for title, prompt in recommended_demo_scenarios.items():
    print("\n" + "#" * 100)
    print(title)
    print("#" * 100)
    print(run_cookie_agent(prompt))


####################################################################################################
Demo Scenario 1
####################################################################################################
You can use the following substitutes for butter in cookies:

1. **Coconut Oil**: This can be used in solid or melted form. If melted, it may increase the spread of your cookies.
   
2. **Plant-Based Butter**: This is a direct alternative that can mimic the flavor and texture of butter but may still alter the spread and final taste slightly.

**Likely Effects**:
- Cookies may spread differently depending on the substitute used.
- The flavor might be less buttery, especially with coconut oil.
- The texture can change depending on whether the substitute is solid or melted—coconut oil, for instance, can create denser or fudgier cookies if used in a melted state.

Keep these effects in mind when making your choice!

###########################################################

# Section 9 — Optional Gradio chat app

**Section 9.1** — Install Gradio

In [ ]:
%pip install -qU gradio

**Section 9.2** — Import Gradio

In [ ]:
import gradio as gr

print("Gradio imported.")

Gradio imported.


**Section 9.3** — Create a simple chat wrapper

In [ ]:
def cookie_chat(message, history, batch_size, texture_goal, dietary_need, pantry_notes):
    """
    Gradio chat wrapper.
    message = newest user message
    history = prior chat history
    """

    combined_message = f"""
User request:
{message}

Recipe preferences:
- Batch size: {batch_size}
- Texture goal: {texture_goal}
- Dietary preference: {dietary_need}
- Pantry notes: {pantry_notes if pantry_notes else "None"}
""".strip()

    return run_cookie_agent(combined_message)

print("Chat wrapper created.")

Chat wrapper created.


**Section 9.4** — Create a pastry-style theme and custom CSS

In [ ]:
import gradio as gr

cookie_theme = gr.themes.Soft(
    primary_hue="rose",
    secondary_hue="stone",
    neutral_hue="zinc"
)

cookie_css = """
@import url('https://fonts.googleapis.com/css2?family=Cormorant+Garamond:wght@500;600;700&family=Libre+Baskerville:wght@400;700&family=Inter:wght@400;500;600&display=swap');

:root {
    color-scheme: light !important;

    --bg-main: #f8f3ee;
    --bg-soft: #efe5dc;
    --card: #fffaf5;
    --border: #d8c8ba;
    --text-main: #4f3a32;
    --text-soft: #7a655d;
    --accent: #8e5d6b;
    --accent-dark: #6d4350;
    --user-bubble: #ead7db;
    --bot-bubble: #fbf4ee;
    --shadow: 0 10px 28px rgba(90, 60, 50, 0.08);
}

/* FORCE LIGHT MODE */
html,
body,
.dark,
.gradio-container,
.gradio-container *,
main,
main *,
section,
section * {
    color-scheme: light !important;
}

html,
body,
.gradio-container {
    background: linear-gradient(180deg, #f8f3ee 0%, #f2e8df 100%) !important;
    font-family: 'Inter', sans-serif !important;
    color: var(--text-main) !important;
}

.gradio-container {
    max-width: 1140px !important;
    margin: 0 auto !important;
    padding-top: 14px !important;
}

/* GENERAL TEXT */
.gradio-container,
.gradio-container p,
.gradio-container span,
.gradio-container div,
.gradio-container label,
.gradio-container textarea,
.gradio-container input,
.gradio-container select {
    color: var(--text-main) !important;
}

/* HEADER */
.vintage-hero {
    background: var(--card) !important;
    border: 1px solid var(--border) !important;
    border-radius: 28px !important;
    padding: 26px 28px 18px 28px !important;
    box-shadow: var(--shadow) !important;
    margin-bottom: 14px !important;
    text-align: center !important;
}

.vintage-hero h1 {
    font-family: 'Cormorant Garamond', serif !important;
    font-size: 3.1rem !important;
    line-height: 1.05 !important;
    color: var(--accent-dark) !important;
    margin-bottom: 8px !important;
    letter-spacing: 0.3px !important;
}

.vintage-hero p {
    font-family: 'Libre Baskerville', serif !important;
    font-size: 0.98rem !important;
    line-height: 1.7 !important;
    color: var(--text-soft) !important;
    margin: 0 !important;
}

.vintage-pill-row {
    display: flex !important;
    flex-wrap: wrap !important;
    gap: 10px !important;
    justify-content: center !important;
    margin: 14px 0 4px 0 !important;
}

.vintage-pill {
    background: #f7ece6 !important;
    border: 1px solid #dbc8bf !important;
    border-radius: 999px !important;
    padding: 9px 15px !important;
    color: #775d55 !important;
    font-size: 0.96rem !important;
    font-weight: 500 !important;
}

/* SIDE CARDS */
.vintage-note,
.vintage-card {
    background: var(--card) !important;
    border: 1px solid var(--border) !important;
    border-radius: 22px !important;
    padding: 16px 17px !important;
    box-shadow: 0 8px 22px rgba(90, 60, 50, 0.06) !important;
    margin-bottom: 12px !important;
}

.vintage-note h3,
.vintage-card h3 {
    font-family: 'Cormorant Garamond', serif !important;
    color: var(--accent-dark) !important;
    font-size: 1.6rem !important;
    margin-bottom: 8px !important;
}

.vintage-note p,
.vintage-card p {
    font-size: 0.98rem !important;
    line-height: 1.75 !important;
    color: var(--text-main) !important;
    margin: 0 !important;
}

/* MAIN GRADIO CONTAINERS */
.gradio-container .block,
.gradio-container .form,
.gradio-container .panel,
.gradio-container .wrap,
.gradio-container .container,
.gradio-container .gr-box,
.gradio-container .gr-form,
.gradio-container .gr-group,
.gradio-container .accordion,
.gradio-container .tabs,
.gradio-container .tabitem {
    background: #fffaf5 !important;
    border-color: var(--border) !important;
    color: var(--text-main) !important;
    border-radius: 18px !important;
}

/* CHATBOT */
#cookie-chatbot,
#cookie-chatbot *,
.gradio-container .chatbot,
.gradio-container .chatbot * {
    background-color: #fffaf5 !important;
    color: var(--text-main) !important;
    border-color: var(--border) !important;
}

#cookie-chatbot {
    border: 1px solid var(--border) !important;
    border-radius: 24px !important;
    box-shadow: var(--shadow) !important;
}

#cookie-chatbot .message,
#cookie-chatbot .bubble,
#cookie-chatbot .prose,
#cookie-chatbot p,
#cookie-chatbot span,
#cookie-chatbot div {
    color: var(--text-main) !important;
}

#cookie-chatbot [data-testid="user"],
#cookie-chatbot .message.user,
#cookie-chatbot .user {
    background: var(--user-bubble) !important;
    color: var(--text-main) !important;
    border-radius: 18px !important;
}

#cookie-chatbot [data-testid="bot"],
#cookie-chatbot .message.bot,
#cookie-chatbot .bot {
    background: var(--bot-bubble) !important;
    color: var(--text-main) !important;
    border-radius: 18px !important;
}

/* TEXTBOXES, DROPDOWNS, INPUTS */
textarea,
input,
select,
.gradio-container textarea,
.gradio-container input,
.gradio-container select,
.gradio-container [role="textbox"],
.gradio-container [data-testid="textbox"],
.gradio-container [data-testid="dropdown"],
.gradio-container .gr-textbox,
.gradio-container .gr-dropdown,
.gradio-container .svelte-select,
.gradio-container .input,
.gradio-container .input-container,
.gradio-container .wrap-inner {
    background: #fffaf5 !important;
    background-color: #fffaf5 !important;
    color: var(--text-main) !important;
    border-color: var(--border) !important;
    border-radius: 16px !important;
}

textarea::placeholder,
input::placeholder {
    color: #8a746c !important;
    opacity: 1 !important;
}

/* DROPDOWN OPTIONS */
.gradio-container [role="listbox"],
.gradio-container [role="option"],
.gradio-container .dropdown,
.gradio-container .dropdown-item,
.gradio-container .options,
.gradio-container .select-menu,
.gradio-container .svelte-select-list,
.gradio-container .item,
.gradio-container .option,
.gradio-container option {
    background: #fffaf5 !important;
    background-color: #fffaf5 !important;
    color: var(--text-main) !important;
    border-color: var(--border) !important;
}

.gradio-container [role="option"]:hover,
.gradio-container .dropdown-item:hover,
.gradio-container .item:hover,
.gradio-container .option:hover {
    background: #f1e3db !important;
    color: var(--text-main) !important;
}

/* EXAMPLES TABLE FIX */
.gradio-container table,
.gradio-container thead,
.gradio-container tbody,
.gradio-container tr,
.gradio-container th,
.gradio-container td,
.gradio-container .table-wrap,
.gradio-container .table,
.gradio-container .examples,
.gradio-container .dataset,
.gradio-container .samples {
    background: #fffaf5 !important;
    background-color: #fffaf5 !important;
    color: var(--text-main) !important;
    border-color: var(--border) !important;
}

.gradio-container th,
.gradio-container td {
    color: var(--text-main) !important;
    border-color: var(--border) !important;
}

.gradio-container tr:hover,
.gradio-container td:hover {
    background: #f1e3db !important;
    color: var(--text-main) !important;
}

/* BUTTONS */
button,
.gr-button {
    border-radius: 16px !important;
    font-size: 0.98rem !important;
}

button.primary,
button[variant="primary"],
.gradio-container button[type="submit"] {
    background: var(--accent) !important;
    border: none !important;
    color: white !important;
}

button.primary:hover,
button[variant="primary"]:hover,
.gradio-container button[type="submit"]:hover {
    background: var(--accent-dark) !important;
    color: white !important;
}

/* LABELS */
label,
.label-wrap,
.label-wrap span,
.gradio-container label,
.gradio-container .label-wrap,
.gradio-container .label-wrap span {
    color: var(--text-main) !important;
    font-weight: 600 !important;
    font-size: 1rem !important;
}

/* HIDE FOOTER */
footer {
    visibility: hidden !important;
}
"""

print("Vintage bakery theme and CSS created.")

Vintage bakery theme and CSS created.


**Section 9.5** — Build the Gradio app

In [ ]:
try:
    demo.close()
except Exception:
    pass


def cookie_chat_manual(message, history, batch_size, texture_goal, dietary_need, pantry_notes):
    """
    Manual Gradio chat wrapper.
    message = newest user message
    history = prior chat history
    """

    if history is None:
        history = []

    if message is None or message.strip() == "":
        return history, ""

    combined_message = f"""
User request:
{message}

Recipe preferences:
- Batch size: {batch_size}
- Texture goal: {texture_goal}
- Dietary preference: {dietary_need}
- Pantry notes: {pantry_notes if pantry_notes else "None"}
""".strip()

    response = run_cookie_agent(combined_message)

    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": response})

    return history, ""


with gr.Blocks(
    title="Cookie Recipe Adaptation Agent",
    fill_width=True
) as demo:

    gr.HTML(
        """
        <div class="vintage-hero">
            <h1>🍪 Cookie Recipe Adaptation Agent 🍪</h1>
            <p>
                A refined bakery-style assistant for cookie substitutions, dietary adjustments,
                batch scaling, and texture guidance . Designed to make recipe changes clearer,
                easier, and more beginner-friendly.
            </p>
            <div class="vintage-pill-row">
                <div class="vintage-pill">🧈 Recipe Retrieval</div>
                <div class="vintage-pill">🥚 Ingredient Scaling</div>
                <div class="vintage-pill">📏 Subsitution Guidance</div>
                <div class="vintage-pill">🍪 Recipe Adaptation</div>
            </div>
        </div>
        """
    )

    with gr.Row():
        with gr.Column(scale=7):

            chatbot = gr.Chatbot(
                elem_id="cookie-chatbot",
                height=330,
                label="Bakery Chat"
            )

            batch_size = gr.Dropdown(
                choices=["Small Batch", "Standard Batch", "Double Batch", "Party Batch"],
                value="Standard Batch",
                label="Batch Size"
            )

            texture_goal = gr.Dropdown(
                choices=["Chewy", "Crispy", "Soft", "Thick Bakery-Style"],
                value="Chewy",
                label="Texture Goal"
            )

            dietary_need = gr.Dropdown(
                choices=["None", "Egg-Free", "Dairy-Free", "Gluten-Free", "Lower Sugar"],
                value="None",
                label="Dietary Preference"
            )

            pantry_notes = gr.Textbox(
                label="Pantry Notes",
                placeholder="Example: I only have salted butter, brown sugar, and oat milk.",
                lines=3
            )

            textbox = gr.Textbox(
                placeholder="Type your cookie question here...",
                label=None,
                lines=3
            )

            bake_btn = gr.Button("Bake", variant="primary")

            gr.Examples(
                examples=[
                    ["What can I use instead of butter in cookies? 🧈", "Standard Batch", "Chewy", "None", ""],
                    ["How do I make a cookie recipe egg-free? 🥚", "Standard Batch", "Soft", "Egg-Free", ""],
                    ["How can I make cookies chewier? 🍪", "Standard Batch", "Chewy", "None", ""],
                    ["""Make this cookie recipe egg-free:

1 cup salted butter softened
1 cup granulated sugar
1 cup light brown sugar packed
2 teaspoons pure vanilla extract
2 large eggs
3 cups all-purpose flour
1 teaspoon baking soda
1/2 teaspoon baking powder
1 teaspoon sea salt
2 cups chocolate chips""",
                     "Standard Batch", "Chewy", "Egg-Free", ""],
                    ["""Scale this cookie recipe from 24 servings to 12 servings.

Recipe ingredients:
1 cup butter | 1/2 cup brown sugar | 2 cups flour | 1 egg | 1 tsp vanilla""",
                     "Small Batch", "Chewy", "None", ""]
                ],
                inputs=[textbox, batch_size, texture_goal, dietary_need, pantry_notes],
                label="Examples"
            )

            textbox.submit(
                fn=cookie_chat_manual,
                inputs=[textbox, chatbot, batch_size, texture_goal, dietary_need, pantry_notes],
                outputs=[chatbot, textbox]
            )

            bake_btn.click(
                fn=cookie_chat_manual,
                inputs=[textbox, chatbot, batch_size, texture_goal, dietary_need, pantry_notes],
                outputs=[chatbot, textbox]
            )

        with gr.Column(scale=3):
            gr.HTML(
                """
                <div class="vintage-note">
                    <h3>What does this agent do?</h3>
                    <p>
                        This assistant is focused on cookie recipes only.
                        It can help with substitutions, batch-size changes, pantry limitations,
                        and simple dietary adjustments while explaining how those changes may
                        affect texture, spread, and overall results.
                    </p>
                </div>
                """
            )

            gr.HTML(
                """
                <div class="vintage-card">
                    <h3>Need a prompt idea?</h3>
                    <p>
                        • Make this recipe dairy-free.<br>
                        • Scale this to a small batch.<br>
                        • I only have brown sugar.<br>
                        • Keep the cookies soft but reduce the sugar.<br>
                        • Replace butter but keep the texture chewy.
                    </p>
                </div>
                """
            )

print("Section 9 app created.")

Closing server running on port: 7862
Section 9 app created.


**Section 9.6** — Launch the app

In [ ]:
demo.launch(
    share=True,
    debug=True,
    theme=cookie_theme,
    css=cookie_css
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://d5ed9753664721a387.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7862 <> https://d5ed9753664721a387.gradio.live


# Section 10 — Final Notes and Limitations

## What This Agent Does Well
This notebook builds a single AI agent for cookie recipe adaptation. The agent can help with:
- ingredient substitutions
- batch size scaling
- dietary adjustments such as egg-free or dairy-free changes
- cookie texture guidance such as chewy, soft, or crispy adjustments

The system combines a small baking knowledge base, recipe retrieval, custom tools, and a LangChain agent workflow to produce practical cookie-focused responses.

## Current Limitations
This version works best when the user gives a self-contained request. For example, the agent performs most reliably when the full recipe or the full question is included in the same prompt.

Follow-up questions without repeated context may be treated like a new input instead of as part of an ongoing conversation. This was a scope decision to keep the final single-agent build simpler and more stable.

The agent is intentionally limited to cookie recipes only. If a user asks about brownies, cakes, or other pastries, the agent may refuse or redirect the request rather than trying to adapt recipes outside its intended scope.

Substitution guidance is helpful but not perfect. Real baking outcomes can still vary depending on ingredient brands, dough temperature, measuring accuracy, oven behavior, and recipe structure. Because of that, the agent should be treated as recipe guidance rather than as a guarantee of identical baking results.

The knowledge base is also intentionally small and focused. It is designed to support common cookie adaptations, not to serve as a complete baking encyclopedia.

## Optional Interface Note
Section 9 includes an optional Gradio interface to present the agent in a more user-friendly way.